## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and connects the data from Google Drive.

**Before you run it**, make sure you've opened the shared camp Drive folder and
clicked **"Add shortcut to Drive"** (put the shortcut in *My Drive*) — that's how
the notebook finds the data file. Then run the cell below and click **Connect** on
the Drive pop-up. Wait for **✅ Setup complete**, then run the rest top to bottom.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os, sys, glob

print("1/3  installing mne ...")
get_ipython().system('pip install -q "mne==1.10.1"')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  connecting Google Drive for the data ...")
from google.colab import drive
drive.mount("/content/drive")
hits = sorted(glob.glob("/content/drive/MyDrive/**/synapse_preprocessed.pkl", recursive=True))
assert hits, (
    "Could not find synapse_preprocessed.pkl in your Drive.\n"
    "Open the shared camp folder, click 'Add shortcut to Drive', put the shortcut "
    "in 'My Drive', then run this cell again."
)
os.environ["CAMP_DATA_PATH"] = hits[0]
os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
print(f"\n\u2705 Setup complete. Using data at: {hits[0]}")
print("Your figures will be saved to Drive > DecodingBrain_outputs.")


# Week 3 · Tier 2 — Subgroup Profiling

**Tier 2 "EEG Statistician"** goes beyond "EXP vs CTRL." This notebook tackles
**Research Goal 4**:

> Are all sound-sensitive people alike in the brain? Or is "hyperacusis" really
> a *spectrum* of overlapping conditions?

The EXP group isn't uniform. We split it three ways:
- **Pure hyperacusis** (n=7): hyperacusis and nothing else
- **Comorbid hyperacusis** (n=7): hyperacusis *plus* tinnitus / hearing loss / misophonia
- **Non-hyperacusis** (n=4): sound-sensitive but not hyperacusis — a *negative control*

### By the end of this notebook you will be able to
1. Split EXP into clinical subgroups
2. Compare a brain feature across subgroups + controls
3. Build a clinical **radar (profile) chart** per subgroup
4. Reason about heterogeneity in a disorder

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import camp_utils as cu

data = cu.load_camp_data(verbose=False)
features = pd.read_csv(cu.save_path("features_table.csv"))

## 1. The subgroups
`cu.SUBGROUPS` holds the membership lists (built from each person's diagnoses).

In [ ]:
for key, members in cu.SUBGROUPS.items():
    print(f"  {cu.SUBGROUP_NAMES[key]:22s} (n={len(members)}): {members}")

## 2. Label each subject with a subgroup
### ✏️ Your turn #1 — write the lookup
Fill in a function that returns which subgroup a subject belongs to (or `"CTRL"`
for controls, or `None` if not found).

In [ ]:
def subgroup_of(subject):
    if subject.startswith("CTRL"):
        return "CTRL"
    # TODO: loop over cu.SUBGROUPS.items(); if subject is in members, return the key
    for key, members in cu.SUBGROUPS.items():
        pass  # replace: if subject in members: return key
    return None

# attach as a column
features["subgroup"] = features["subject"].apply(subgroup_of)
print(features["subgroup"].value_counts(dropna=False))

cu.check(features["subgroup"].notna().all()
         and (features["subgroup"] == "pure_hyperacusis").sum() == 7,
         "Every subject got a subgroup label.",
         "Inside the loop: `if subject in members: return key`.")

## 3. Compare a feature across all groups
Now we can see whether the subgroups differ from each other and from controls.
We'll plot `let_gamma` for: CTRL, non-hyperacusis, comorbid, pure.

In [ ]:
order = ["CTRL", "non_hyperacusis", "comorbid_hyperacusis", "pure_hyperacusis"]
labels = {"CTRL": "Control", **cu.SUBGROUP_NAMES}
colors = {"CTRL": cu.CTRL_COLOR, **cu.SUBGROUP_COLORS}

feature = "let_gamma"
fig, ax = plt.subplots(figsize=(8, 5))
for i, grp in enumerate(order):
    vals = features.loc[features["subgroup"] == grp, feature].dropna().values
    jitter = np.random.uniform(-0.08, 0.08, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, color=colors[grp],
               s=70, alpha=0.8, zorder=3)
    if len(vals):
        ax.plot([i - 0.2, i + 0.2], [vals.mean()] * 2, color="black", lw=2)  # mean bar
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([labels[g] for g in order], rotation=15)
ax.set_ylabel(f"{feature} (dB)")
ax.set_title("Does the brain feature change across subgroups?")
plt.tight_layout()
plt.savefig(cu.save_path("tier2_subgroups.png"), dpi=300, bbox_inches="tight")
plt.show()

**Read it like a scientist:** Does the *negative control* (non-hyperacusis) sit
closer to the real controls? Do *pure* hyperacusis cases show the strongest
effect? Those patterns would support "hyperacusis is its own thing."

### ✏️ Your turn #2 — quantify it
Print the mean of your chosen feature for each subgroup, in `order`, and say
which subgroup is furthest from the controls.

In [ ]:
means = {}
for grp in order:
    vals = features.loc[features["subgroup"] == grp, feature].dropna()
    means[grp] = vals.mean()
    print(f"  {labels[grp]:22s}: {means[grp]:+.2f} dB")

# TODO: which subgroup's mean is furthest from the CTRL mean?
#   hint: compare abs(means[grp] - means["CTRL"]) across the hyperacusis groups
furthest = None

cu.check(furthest in cu.SUBGROUPS,
         f"Furthest-from-control subgroup: {labels.get(furthest, furthest)}",
         "Pick the subgroup key with the largest abs(mean - CTRL mean).")

## 4. Clinical radar (profile) charts
A **radar chart** shows several scores at once as a shape. We'll draw the average
symptom profile of each subgroup across four questionnaires — a quick way to see
*how* the subgroups differ clinically.

In [ ]:
profile_measures = ["HQ_Total", "GAD_Total", "THI_Total", "Miso_Section1"]

def subgroup_profile(data, members):
    """Average clinical score per measure for a list of subjects."""
    out = []
    for m in profile_measures:
        vals = [cu.get_clinical_score(data, s, m) for s in members]
        vals = [v for v in vals if v is not None]
        out.append(np.mean(vals) if vals else 0.0)
    return out

# angles for the radar (one spoke per measure, looped closed)
angles = np.linspace(0, 2 * np.pi, len(profile_measures), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
for key, members in cu.SUBGROUPS.items():
    vals = subgroup_profile(data, members)
    vals += vals[:1]  # close the loop
    ax.plot(angles, vals, color=cu.SUBGROUP_COLORS[key],
            label=cu.SUBGROUP_NAMES[key], linewidth=2)
    ax.fill(angles, vals, color=cu.SUBGROUP_COLORS[key], alpha=0.1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(profile_measures)
ax.set_title("Clinical profiles by subgroup", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(cu.save_path("tier2_radar.png"), dpi=300, bbox_inches="tight")
plt.show()

## 🎯 Wrap-up (Tier 2)
You discovered that a single diagnosis label can hide very different people. By
splitting into subgroups — and using a negative control — you can ask whether a
brain signature is *specific* to hyperacusis or shared across sound disorders.

**Caution:** these subgroups are tiny (n=4–7). Treat everything here as
exploratory and say so on your poster.

➡️ **Next (Tier 2):** Notebook 12 — *when* in time do group differences appear?